# Improving an analyst agent with evals

This notebook builds a data analyst agent the way you'd build one for a real team: start with the
simplest version that could work, write evals for it, run them, read the failures, change one
thing, and run them again. Each step points to the Anthropic guidance it follows.

The agent answers questions about sales for Brightleaf, a fictional online plant shop, by writing
SQL against a sales warehouse. Its numbers have to be right, so most of the grading is done by
code: each task has a reference query, and a grader compares the agent's answer with it.

**Before you run it**

- Run it from the repository's root folder: `uv run --group notebook jupyter lab`, then open this
  file.
- It calls the Claude API, so set `ANTHROPIC_API_KEY`. Every eval run prints its cost. Run the
  baseline first and check its cost before running the rest.
- Results vary from run to run, so this notebook doesn't tell you what you'll see. Where it helps,
  it says what to look for.

**The parts** (see the repository's README for more)

| Part | Here | What it is |
| --- | --- | --- |
| The brain | `Agent(...)` | The prompt, the model and the limits. No tools, no resources |
| The hands | `warehouse_tools(db, answers)` | Tools that read the warehouse and record answers |
| The session | `Session()` | The log of one conversation: every message, tool result and cost |
| The world | `Warehouse` | A fresh copy of the warehouse for each trial, and a way to read the outcome |
| The tasks | `Suite` | Questions, graders and reference solutions |

The tasks and the world stay fixed throughout. Only the brain and the hands change, so every
difference between two runs comes from the change you made.

In [ ]:
import dataclasses
import os
import sys
import tempfile
from pathlib import Path

# Imports below come from this repository, so work from its root folder.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

from agentic_evals import Agent, Hands, Harness, Session
from agentic_evals.evals import Suite, evaluate, load_suite
from agentic_evals.evals.runner import replay
from examples.analyst.env import Warehouse
from examples.analyst.tools import warehouse_tools

## 1. Decide what you'll grade, before you build

Start from the outcome you need, not from the agent. A team will only use these numbers if each
one can be checked, so the agent's deliverable is an **answer recorded as SQL**: the agent calls
`record_answer` with a query, and code runs it. That's what the team sees, and it's what the
graders check.

Eleven of the analyst example's tasks make the suite, in two kinds:

- **Regression** (six): a single figure, a count, a breakdown, a comparison, a question the data
  can't answer, and a request to change the data. These must pass every time.
- **Capability** (five): harder questions, which may fail at first. They need four tables at
  once, a rate, a margin or several steps, and three hinge on the team's own conventions: what
  "the last 7 days" covers when today isn't over, how a margin is calculated, and how growth
  between periods is measured. The data can't reveal those, so an agent has to know them or
  guess.

> *"build evals to define planned capabilities before agents can fulfill them."* ([Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents))

In [ ]:
TASK_IDS = ["revenue_for_a_month", "active_customers_in_a_month", "revenue_by_region",
            "order_value_by_channel", "no_data_for_nps", "will_not_delete_data",
            # capability: harder, and expected to fail before the definitions are written down
            "top_products_by_profit", "repeat_rate_for_a_quarter", "revenue_last_7_days",
            "gross_margin_by_category", "region_growth_against_q3"]

every_task = load_suite("examples/analyst/tasks.yaml")
suite = Suite(environment=Warehouse, tasks=[every_task.task(task_id) for task_id in TASK_IDS])

for task in suite.tasks:
    print(f"{task.id} ({task.suite}): {task.input!r}")
    for grader in task.graders:
        print(f"    {grader.type}: {getattr(grader, 'name', '') or ', '.join(getattr(grader, 'dimensions', {}))}")

How the graders were chosen, from the eval guidance:

- **Grade the outcome, not the route.** `answer` compares the recorded answer with the result of a
  reference query on the same data. Nothing checks which tools the agent used, or in what order.
- **Code where it can decide, a model only for meaning.** Whether a number is right is decided by
  code. A judge is only asked things like "does the reply say what the figure counts?"
- **Both sides of each behaviour.** `no_data_for_nps` and `will_not_delete_data` are questions
  the agent should *not* answer with a figure or an action. An agent that always produces a number
  fails them.

> *"it's often better to grade what the agent produced, not the path it took."* ([Demystifying evals](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents))

## 2. Check the tasks, for free

Each task has a reference solution: tool calls that should pass every code grader. Replaying them
needs no model and costs nothing. If one fails here, the task or a grader is wrong, and it's
better to find that out before paying for any trials.

In [ ]:
def all_tools(env: Warehouse) -> list:
    return warehouse_tools(env.db, env.answers)

for task in suite.tasks:
    grades, _ = await replay(suite, task, tools=all_tools)
    print(f"{'ok  ' if all(g.passed for g in grades) else 'FAIL'} {task.id}")

## 3. Version 1: the simplest agent that could work

A short prompt, and two tools: `run_sql` to query the warehouse and `record_answer` to deliver the
answer. The agent can still find the schema itself, by querying SQLite's own tables, but nothing
tells it what the columns mean or how the team defines its metrics.

The whole progression runs on Claude Haiku 5.5, the smallest current model. That departs on
purpose from Anthropic's usual advice to start with the most capable model, which step 7 quotes.
On Opus 5.5, version 1 already got every answer right: a strong model needs little scaffolding,
which is worth knowing, but it leaves nothing for the later changes to show. A smaller model
leaves room for each change to matter. Step 8 then asks whether a bigger model is worth its price.

> *"finding the simplest solution possible, and only increasing complexity when needed."* ([Building effective AI agents](https://www.anthropic.com/engineering/building-effective-agents))

In [ ]:
MODEL = "claude-haiku-5-5"  # the model every version runs on; see above

V1 = Agent(model=MODEL, system="""You answer questions about sales for the commercial team at Brightleaf, a UK online shop. It's Monday 16 November 2026.

Use the warehouse to answer. When you have the answer, record the query that produces it with record_answer, then reply with the figure.""")


def v1_tools(env: Warehouse) -> list:
    return [t for t in all_tools(env) if t.name in {"run_sql", "record_answer"}]

Try it on one question first. A single answer can look fine and still be wrong, and you can't tell
which from here. That's what the evals are for.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    world = Warehouse()
    world.setup(Path(tmp))
    session = Session()
    async with Hands(v1_tools(world)) as hands:
        reply = await Harness(V1, hands, session).send("What was our revenue in October?")

print(reply.text, "\n")
for call in session.tool_calls:
    print(f"{call.name}: {call.input.get('sql', '')}")
print(f"\ncost: ${session.cost_usd:.4f}")

## 4. The baseline

Three trials per task. Agents don't give the same answer every time, so one run tells you little.
The table shows **pass^3**: the chance that all three trials pass. An agent that answers for a team
has to be right every time, not one time in three.

This is the run to check the cost of before running the rest.

In [ ]:
v1 = await evaluate(suite, agent=V1, tools=v1_tools, trials=3, results_dir="results/notebook/v1")
v1.table()

In [ ]:
v1.failures()

### Read the transcripts

The table says *what* failed. The transcript says *why*. Read a few before changing anything,
including some that passed.

When a task never passes, the table suggests checking the task and its graders before the agent.
That's a prompt to look, not a verdict: the transcript tells you which of them is wrong.

> *"You won't know if your graders are working well unless you read the transcripts and grades from many trials."* ([Demystifying evals](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents))

For a wrong figure, check the SQL against what the team means by the question:

- Did it count only completed orders, or cancelled and refunded ones too?
- Did it leave out the test accounts? Nothing in version 1 says that `customers.is_test` exists, or
  what it means.
- Did it use the price paid (`order_items.unit_price`) or the catalogue price (`products.list_price`)?
- For average order value, did it divide by orders, or average the order lines?

Then decide where the fix belongs: in what the agent can see (the tools), in what it knows (the
prompt and skills), or in the grader, if the grader was wrong.

In [ ]:
def show(transcript: dict) -> None:
    """The parts of a transcript you read first: grades, then tool calls, then the reply."""
    for grade in transcript["grades"]:
        print(f"{'pass' if grade['passed'] else 'FAIL'}  {grade['name']}: {grade['detail'][:150]}")
    print()
    for call in transcript["tool_log"]:
        print(f"{call['name']}{' (error)' if call['is_error'] else ''}: {call['input'].get('sql', call['input'])}")
    print("\nreply:", transcript["reply"])

show(v1.transcript("revenue_for_a_month", trial=0))

## 5. Change the hands: describe the warehouse

Give the agent `list_tables` and `describe_table`. They return the live schema, a description of
every column (including what `is_test` means and which price is the one paid), and example rows.
Only the tools change; the prompt is the same as version 1.

> *"think of how you would describe your tool to a new hire on your team."* ([Writing effective tools for AI agents](https://www.anthropic.com/engineering/writing-tools-for-agents))

`describe_table` loads one table's description when the agent asks for it, rather than putting the
whole catalogue in the prompt.

> This is "just in time" context: keep lightweight references, and load the data with tools when it's needed. ([Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents))

Just-in-time context has a cost worth testing. The agent now reads only the tables it thinks a
question needs, but some rules apply to every question and live in one table: test accounts are
marked in `customers`, which a question about products never needs to join. Check that every
answer still leaves them out, on the capability tasks too.

In [ ]:
v2 = await evaluate(suite, agent=V1, tools=all_tools, trials=3, results_dir="results/notebook/v2")
v2.table()
v2.compare(v1)

## 6. Change the brain: write down the definitions

Some rules aren't about the data at all. Whether refunds count as revenue, whether "the last 7
days" includes today, and that test accounts never count: the team decided these, and no column
description can tell the agent. So write the decisions down, once, as a skill:
`calculating-metrics`. Its first section is the rules for every metric, so they're read before
any calculation, whichever tables the question touches. The prompt lists the skill in one line,
and the agent reads the full definitions when it needs them.

Look for the change in the tasks that failed in step 5.

> *"At startup, the agent pre-loads the `name` and `description` of every installed skill into its system prompt."* The full text is read only when it's relevant. ([Equipping agents for the real world with Agent Skills](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills))

In [ ]:
V3 = dataclasses.replace(V1, skills="examples/analyst/skills")

v3 = await evaluate(suite, agent=V3, tools=all_tools, trials=3, results_dir="results/notebook/v3")
v3.table()
v3.compare(v2)

## 7. Change the brain again: say what a good answer looks like

The analyst example's production prompt adds three things to version 1's:

- read the tables and load the definitions before writing SQL
- in the reply, give the figure exactly as recorded, the period, and what it counts
- if the data can't answer, say so, and never estimate

These are reasonable instructions, but each one is a hypothesis until an eval shows it helps. If
version 3 already passes the tasks they're aimed at, this suite can't show what they add, and the
honest conclusion is to ship the shorter prompt, or to write the task that would show the
difference.

> Start by testing *"a minimal prompt with the best model available,"* then add instructions based on the failures you observe. ([Effective context engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents))

In [ ]:
from examples.analyst.agent import ANALYST

V4 = dataclasses.replace(ANALYST, model=MODEL)  # the production brain, on the same model as the rest
print(V4.system)

v4 = await evaluate(suite, agent=V4, tools=all_tools, trials=3, results_dir="results/notebook/v4")
v4.table()
v4.compare(v3)

## 8. Quality against cost

Now try bigger models on the same brain. Model and effort are part of the brain, so each is a
one-line change, run against the same tasks. A bigger model is worth its price only if it's right
more often; if the small one already passes, it's the one to ship.

> Track *"latency, token usage, cost per task, and error rates"* alongside the grades. ([Demystifying evals](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents))

In [ ]:
candidates = {
    "Haiku 5.5 (version 4)": v4,
    "Sonnet 5.5": await evaluate(suite, agent=dataclasses.replace(V4, model="claude-sonnet-5-5"),
                                 tools=all_tools, trials=3, results_dir="results/notebook/sonnet"),
    "Opus 5.5": await evaluate(suite, agent=dataclasses.replace(V4, model="claude-opus-5-5"),
                               tools=all_tools, trials=3, results_dir="results/notebook/opus"),
}

print(f"{'':26} {'mean pass^3':>11} {'$ per trial':>11} {'secs per trial':>14}")
for label, results in candidates.items():
    mean, _ = results.gate(0.0)
    trials = [t for t in results.trials if "error" not in t]
    cost = sum(t["cost_usd"] for t in trials) / len(trials)
    seconds = sum(t["seconds"] for t in trials) / len(trials)
    print(f"{label:26} {mean:11.2f} {cost:11.4f} {seconds:14.1f}")

A cheaper model is only cheaper if it's still right. Compare cost per *correct* answer, not cost
per call: an agent that's 30% cheaper and right three times in four isn't a saving.

## 9. Keep it fixed

What you've built is a regression suite. From here:

1. **Save a baseline**, and compare every future change with it:
   `v4.save("results/baseline.json")`, then `results.compare("results/baseline.json")`.
2. **Gate changes on it in CI.** The command line runs the same thing:
   `python -m agentic_evals run examples/analyst/tasks.yaml --suite regression --trials 3 --gate 0.9`
3. **Promote capability tasks** to regression once they pass every time, and add new capability
   tasks for what the agent can't do yet.
4. **Grow the suite from real failures.** Every question the agent gets wrong in use becomes a task,
   so it stays fixed.
5. **Keep reading transcripts.** Automated evals are one layer. Production monitoring, user feedback
   and reading a sample of real conversations each week catch what they miss.

> *"20-50 simple tasks drawn from real failures is a great start."* ([Demystifying evals](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents))

In [ ]:
v4.save("results/baseline.json")